In [ ]:
# Movie Recommendation System
## Comparative Analysis of Recommendation Algorithms

### Algorithms Used
#- CountVectorizer + Cosine Similarity
#- TF-IDF + Cosine Similarity
#- K-Nearest Neighbors (KNN)

### Dataset
#- movie_dataset.csv

In [1]:
import pandas as pd
import numpy as np
import ast
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

import nltk
from nltk.stem.porter import PorterStemmer

In [2]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Smruti\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
movies = pd.read_csv("../data/movie_dataset.csv")
movies.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [4]:
movies.shape

(4803, 24)

In [5]:
movies.columns

Index(['index', 'budget', 'genres', 'homepage', 'id', 'keywords',
       'original_language', 'original_title', 'overview', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'vote_average', 'vote_count', 'cast', 'crew', 'director'],
      dtype='object')

In [6]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

In [7]:
movies.isnull().sum()

index                      0
budget                     0
genres                    28
homepage                3091
id                         0
keywords                 412
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
cast                      43
crew                       0
director                  30
dtype: int64

In [8]:
movies = movies[['title', 'genres', 'keywords', 'cast', 'director', 'overview']]
movies.head()

,title,genres,keywords,cast,director,overview
0,Avatar,Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,Sam Worthington Zoe Saldana Sigourney Weaver S...,James Cameron,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,Johnny Depp Orlando Bloom Keira Knightley Stel...,Gore Verbinski,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,Action Adventure Crime,spy based on novel secret agent sequel mi6,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,Sam Mendes,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,Christian Bale Michael Caine Gary Oldman Anne ...,Christopher Nolan,Following the death of District Attorney Harve...
4,John Carter,Action Adventure Science Fiction,based on novel mars medallion space travel pri...,Taylor Kitsch Lynn Collins Samantha Morton Wil...,Andrew Stanton,"John Carter is a war-weary, former military ca..."


In [9]:
movies.dropna(inplace=True)
movies.isnull().sum()

title       0
genres      0
keywords    0
cast        0
director    0
overview    0
dtype: int64

In [10]:
movies.duplicated().sum()

0

In [11]:
def convert(text):
    try:
        L = []
        for i in ast.literal_eval(text):
            L.append(i['name'])
        return L
    except:
        return []

In [12]:
def convert3(text):
    try:
        L = []
        counter = 0
        for i in ast.literal_eval(text):
            if counter != 3:
                L.append(i['name'])
                counter += 1
            else:
                break
        return L
    except:
        return []

In [13]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['overview'] = movies['overview'].apply(lambda x: x.split())
movies['director'] = movies['director'].apply(lambda x: [x])

In [14]:
movies.head()

,title,genres,keywords,cast,director,overview
0,Avatar,[],[],[],[James Cameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,[],[],[],[Gore Verbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,[],[],[],[Sam Mendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,[],[],[],[Christopher Nolan],"[Following, the, death, of, District, Attorney..."
4,John Carter,[],[],[],[Andrew Stanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [15]:
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ", "") for i in x])
movies['director'] = movies['director'].apply(lambda x: [i.replace(" ", "") for i in x])

In [16]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['director']

In [17]:
new_df = movies[['title', 'tags']]
new_df.head()

,title,tags
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."


In [18]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())
new_df.head()

,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


In [19]:
ps = PorterStemmer()

In [20]:
def stem(text):
    return " ".join([ps.stem(word) for word in text.split()])

In [21]:
new_df['tags'] = new_df['tags'].apply(stem)
new_df.head()

,title,tags
0,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,Spectre,a cryptic messag from bond’ past send him on a...
3,The Dark Knight Rises,follow the death of district attorney harvey d...
4,John Carter,"john carter is a war-weary, former militari ca..."


In [22]:
## MODEL 1 — CountVectorizer + Cosine Similarity

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors_count = cv.fit_transform(new_df['tags']).toarray()
vectors_count.shape

(4375, 5000)

In [23]:
similarity_count = cosine_similarity(vectors_count)
similarity_count


array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.03419928, 0.03984095,
        0.04303315],
       [0.        , 0.        , 1.        , ..., 0.        , 0.03289758,
        0.        ],
       ...,
       [0.        , 0.03419928, 0.        , ..., 1.        , 0.        ,
        0.15452877],
       [0.        , 0.03984095, 0.03289758, ..., 0.        , 1.        ,
        0.02571722],
       [0.        , 0.04303315, 0.        , ..., 0.15452877, 0.02571722,
        1.        ]])

In [24]:
def recommend_count(movie):
    if movie not in new_df['title'].values:
        return ["Movie not found"]

    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity_count[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    recommendations = []
    for i in movies_list:
        recommendations.append(new_df.iloc[i[0]].title)

    return recommendations

In [25]:
recommend_count("Avatar")

['Bucky Larson: Born to Be a Star',
 'Apollo 18',
 'Tears of the Sun',
 'Aliens vs Predator: Requiem',
 'Meet Dave']

In [26]:
## MODEL 2 — TF-IDF + Cosine Similarity

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vectors_tfidf = tfidf.fit_transform(new_df['tags']).toarray()
vectors_tfidf.shape

(4375, 5000)

In [27]:
similarity_tfidf = cosine_similarity(vectors_tfidf)
similarity_tfidf

array([[1.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.01087042, 0.02811962,
        0.0111667 ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.02007222,
        0.        ],
       ...,
       [0.        , 0.01087042, 0.        , ..., 1.        , 0.        ,
        0.04111489],
       [0.        , 0.02811962, 0.02007222, ..., 0.        , 1.        ,
        0.01055725],
       [0.        , 0.0111667 , 0.        , ..., 0.04111489, 0.01055725,
        1.        ]])

In [28]:
def recommend_tfidf(movie):
    if movie not in new_df['title'].values:
        return ["Movie not found"]

    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity_tfidf[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    recommendations = []
    for i in movies_list:
        recommendations.append(new_df.iloc[i[0]].title)

    return recommendations

In [29]:
recommend_tfidf("Avatar")

['Apollo 18',
 'Aliens',
 'Tears of the Sun',
 'The Adventures of Pluto Nash',
 'Life During Wartime']

In [30]:
## MODEL 3 — KNN
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(vectors_tfidf)

NearestNeighbors(algorithm='brute', metric='cosine')

In [31]:
def recommend_knn(movie):
    if movie not in new_df['title'].values:
        return ["Movie not found"]

    movie_index = new_df[new_df['title'] == movie].index[0]
    movie_vector = vectors_tfidf[movie_index].reshape(1, -1)

    distances, indices = knn.kneighbors(movie_vector, n_neighbors=6)

    recommendations = []
    for i in indices[0][1:]:
        recommendations.append(new_df.iloc[i].title)

    return recommendations

In [32]:
recommend_knn("Avatar")

['Apollo 18',
 'Aliens',
 'Tears of the Sun',
 'The Adventures of Pluto Nash',
 'Life During Wartime']

In [33]:
## MODEL COMPARISON

test_movie = "Avatar"

print("CountVectorizer Recommendations:")
print(recommend_count(test_movie))

print("\nTF-IDF Recommendations:")
print(recommend_tfidf(test_movie))

print("\nKNN Recommendations:")
print(recommend_knn(test_movie))


CountVectorizer Recommendations:
['Bucky Larson: Born to Be a Star', 'Apollo 18', 'Tears of the Sun', 'Aliens vs Predator: Requiem', 'Meet Dave']

TF-IDF Recommendations:
['Apollo 18', 'Aliens', 'Tears of the Sun', 'The Adventures of Pluto Nash', 'Life During Wartime']

KNN Recommendations:
['Apollo 18', 'Aliens', 'Tears of the Sun', 'The Adventures of Pluto Nash', 'Life During Wartime']


In [34]:
## Compare More Movies
test_movies = ["Avatar", "Batman Begins", "The Dark Knight", "Interstellar", "Titanic"]

for movie in test_movies:
    print("="*60)
    print(f"Movie: {movie}")
    print("- Count :", recommend_count(movie))
    print("- TF-IDF:", recommend_tfidf(movie))
    print("- KNN   :", recommend_knn(movie))
    print("="*60)
    print()

Movie: Avatar
- Count : ['Bucky Larson: Born to Be a Star', 'Apollo 18', 'Tears of the Sun', 'Aliens vs Predator: Requiem', 'Meet Dave']
- TF-IDF: ['Apollo 18', 'Aliens', 'Tears of the Sun', 'The Adventures of Pluto Nash', 'Life During Wartime']
- KNN   : ['Apollo 18', 'Aliens', 'Tears of the Sun', 'The Adventures of Pluto Nash', 'Life During Wartime']

Movie: Batman Begins
- Count : ['Auto Focus', 'Alfie', 'Raging Bull', 'Night at the Museum', 'Inkheart']
- TF-IDF: ['Night at the Museum', 'The Island', 'House of Wax', 'The Relic', 'Night at the Museum: Secret of the Tomb']
- KNN   : ['Night at the Museum', 'The Island', 'House of Wax', 'The Relic', 'Night at the Museum: Secret of the Tomb']

Movie: The Dark Knight
- Count : ['The Dark Knight Rises', 'Batman Begins', 'Batman Returns', 'Batman Forever', "Gangster's Paradise: Jerusalema"]
- TF-IDF: ['The Dark Knight Rises', 'Batman Returns', 'Batman Forever', 'Batman Begins', 'Batman: The Dark Knight Returns, Part 2']
- KNN   : ['The Dar

In [35]:
## Manual Comparison Table
comparison_results = pd.DataFrame({
    "Model": [
        "CountVectorizer + Cosine Similarity",
        "TF-IDF + Cosine Similarity",
        "KNN"
    ],
    "Strength": [
        "Simple and effective",
        "More meaningful word weighting",
        "Nearest-neighbor based recommendation"
    ],
    "Weakness": [
        "Common words may dominate",
        "Slightly more computationally heavy",
        "Often similar to TF-IDF output"
    ],
    "Recommendation Quality": [
        "Good",
        "Very Good",
        "Good"
    ]
})

comparison_results

,Model,Strength,Weakness,Recommendation Quality
0,CountVectorizer + Cosine Similarity,Simple and effective,Common words may dominate,Good
1,TF-IDF + Cosine Similarity,More meaningful word weighting,Slightly more computationally heavy,Very Good
2,KNN,Nearest-neighbor based recommendation,Often similar to TF-IDF output,Good


In [37]:
# Final Model Selection

#After comparing all three algorithms, the selected final model is:

## TF-IDF + Cosine Similarity

### Reason:
#- Better semantic relevance
#- Better word importance handling
#- More meaningful recommendations
#- Best suited for metadata-based movie recommendation

In [38]:
# Final Recommendation Function

def recommend(movie):
    return recommend_tfidf(movie)

In [39]:
recommend("Avatar")

['Apollo 18',
 'Aliens',
 'Tears of the Sun',
 'The Adventures of Pluto Nash',
 'Life During Wartime']

In [40]:
recommend("Titanic")

['Ghost Ship',
 'Event Horizon',
 'Dear Frankie',
 'I Can Do Bad All By Myself',
 'Niagara']

In [41]:
pickle.dump(new_df, open("../outputs/movies.pkl", "wb"))
pickle.dump(similarity_tfidf, open("../outputs/similarity.pkl", "wb"))

In [42]:
print("Files saved successfully!")

Files saved successfully!
